# Etapa 04 V2 — Vila do Chaves

Gera 18 personagens, no mínimo 12.000 transações de fundo e eventos narrativos, preservando as seis fontes, três lotes, dados inválidos, hash, auditoria e idempotência da arquitetura original. **A V1 não é sobrescrita.**

Execute na branch `chore/estrutura-inicial` depois de Pull. A execução publica em `cenarios/v2/<data_referencia>`, sem criar tabelas alternativas. O consumo Bronze será separado por lote.

In [0]:
dbutils.widgets.text('catalogo', spark.catalog.currentCatalog())
dbutils.widgets.text('data_referencia', '2026-01-15')
dbutils.widgets.text('batch_id', '')
print('Confirme catalogo=workspace e data_referencia=2026-01-15.')

In [0]:
from pathlib import Path
import sys
atual=Path.cwd().resolve()
raiz=next((p for p in [atual,*atual.parents] if (p/'src'/'medallion').is_dir() and (p/'pyproject.toml').is_file()),None)
if raiz is None:
    raise RuntimeError('Abra o notebook dentro da pasta Git do projeto')
if str(raiz/'src') not in sys.path:
    sys.path.insert(0,str(raiz/'src'))
from medallion.config import PipelineConfig
from medallion.synthetic.execute_v2 import executar_geracao_v2
from medallion.synthetic.generator_v2 import gerar_lotes_v2

In [0]:
parametros={'ambiente':'dev','modo':'incremental','catalogo':dbutils.widgets.get('catalogo').strip(),'data_referencia':dbutils.widgets.get('data_referencia').strip()}
batch_id=dbutils.widgets.get('batch_id').strip()
if batch_id:
    parametros['batch_id']=batch_id
config=PipelineConfig(**parametros)
lotes=gerar_lotes_v2(config.data_referencia)
print('Clientes distintos na V2:',len({c['id_cliente'] for lote in lotes.values() for c in lote['clientes_cdc']}))
print('Transações totais na V2:',sum(len(lote['transacoes']) for lote in lotes.values()))
print('Volume:',config.caminho_entrada)

In [0]:
resultado_v2=executar_geracao_v2(spark,config,raiz)

## Evidências

Confirme `DADOS SINTÉTICOS V2: OK`, os três lotes, o hash e o registro `SUCESSO`. Faça exportação IPYNB com outputs e commit na branch `chore/estrutura-inicial`. Não apague nem sobrescreva o cenário V1. A camada Bronze deverá ler explicitamente `cenarios/v2` e processar as chegadas `01_inicial`, `02_incremental`, `03_atrasado` em execuções separadas.